# 03 — Critical Path Analysis

**Purpose:** Identify the longest (most expensive) path through the build
dependency DAG and quantify per-target slack.  
The critical path determines the minimum wall-clock build time under
unlimited parallelism. Targets on the critical path are prime optimisation
candidates.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from build_optimiser.graph import load_graph, attach_metrics
from build_optimiser.config import load_config

cfg = load_config(PROJECT_ROOT / "config.yaml")

# Load processed data
file_metrics = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "file_metrics.parquet")
target_metrics = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "target_metrics.parquet")
edge_list = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "edge_list.parquet")

# Load dependency graph
G = load_graph(str(PROJECT_ROOT / "data" / "raw" / "dot"))
attach_metrics(G, target_metrics)

print(f"Files: {len(file_metrics)}, Targets: {len(target_metrics)}, Edges: {len(edge_list)}")

In [ ]:
# ── Compute critical path using nx.dag_longest_path ──────────────────
# Weight edges by the compile time of the *target* (destination) node.

assert nx.is_directed_acyclic_graph(G), "Graph must be a DAG for critical-path analysis"

# Assign node weights as compile_time_s (default to 0 if missing)
for node in G.nodes():
    ct = G.nodes[node].get("compile_time_s", 0)
    G.nodes[node]["weight"] = float(ct) if ct and not np.isnan(float(ct)) else 0.0

# nx.dag_longest_path with weight on nodes:
# We transfer node weight to incoming edges so we can use the edge-weight API.
W = G.copy()
for u, v in W.edges():
    W.edges[u, v]["weight"] = W.nodes[v].get("weight", 0.0)

critical_path = nx.dag_longest_path(W, weight="weight")
critical_path_length = nx.dag_longest_path_length(W, weight="weight")

print(f"Critical path length (weighted): {critical_path_length:.2f} s")
print(f"Critical path nodes: {len(critical_path)}")
print()

cp_df = pd.DataFrame({
    "target": critical_path,
    "compile_time_s": [G.nodes[n].get("weight", 0.0) for n in critical_path],
    "position": range(len(critical_path)),
})
cp_df["cumulative_s"] = cp_df["compile_time_s"].cumsum()
print(cp_df.to_string(index=False))

In [ ]:
# ── Slack analysis per target ────────────────────────────────────────
# Slack = (latest start) - (earliest start).  A target on the critical
# path has zero slack.

topo_order = list(nx.topological_sort(G))
node_weight = {n: G.nodes[n].get("weight", 0.0) for n in G.nodes()}

# Forward pass — earliest start (ES) and earliest finish (EF)
es = {n: 0.0 for n in G.nodes()}
ef = {n: 0.0 for n in G.nodes()}
for node in topo_order:
    for pred in G.predecessors(node):
        es[node] = max(es[node], ef[pred])
    ef[node] = es[node] + node_weight[node]

project_duration = max(ef.values())

# Backward pass — latest start (LS) and latest finish (LF)
lf = {n: project_duration for n in G.nodes()}
ls = {n: project_duration for n in G.nodes()}
for node in reversed(topo_order):
    for succ in G.successors(node):
        lf[node] = min(lf[node], ls[succ])
    ls[node] = lf[node] - node_weight[node]

slack = {n: ls[n] - es[n] for n in G.nodes()}

slack_df = pd.DataFrame({
    "target": list(slack.keys()),
    "slack_s": list(slack.values()),
    "es": [es[n] for n in slack],
    "ef": [ef[n] for n in slack],
    "ls": [ls[n] for n in slack],
    "lf": [lf[n] for n in slack],
    "compile_time_s": [node_weight[n] for n in slack],
})
slack_df = slack_df.sort_values("slack_s").reset_index(drop=True)

print(f"Project duration (critical-path): {project_duration:.2f} s")
print(f"Targets with zero slack (on critical path): {(slack_df['slack_s'].abs() < 1e-9).sum()}")
print()

# Slack distribution
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(slack_df["slack_s"], bins=60, edgecolor="black", alpha=0.7, color="steelblue")
ax.set_xlabel("Slack (s)")
ax.set_ylabel("Count")
ax.set_title("Slack distribution across all targets")
ax.axvline(0, color="red", linestyle="--", label="zero slack (critical)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Visualisation of DAG with critical path highlighted ──────────────
# For large graphs, show a subgraph around the critical path.

cp_set = set(critical_path)

# Collect 1-hop neighbours of critical path nodes (capped for readability)
NEIGHBOUR_CAP = 200
neighbourhood = set(critical_path)
for n in critical_path:
    neighbourhood.update(G.predecessors(n))
    neighbourhood.update(G.successors(n))
    if len(neighbourhood) > NEIGHBOUR_CAP:
        break

sub = G.subgraph(neighbourhood).copy()

# Colour nodes
node_colors = ["red" if n in cp_set else "lightblue" for n in sub.nodes()]
node_sizes = [80 if n in cp_set else 20 for n in sub.nodes()]

# Colour edges
cp_edges = set(zip(critical_path[:-1], critical_path[1:]))
edge_colors = ["red" if (u, v) in cp_edges else "grey" for u, v in sub.edges()]
edge_widths = [2.0 if (u, v) in cp_edges else 0.3 for u, v in sub.edges()]

fig, ax = plt.subplots(figsize=(16, 10))
pos = nx.nx_agraph.graphviz_layout(sub, prog="dot") if len(sub) < 300 else nx.spring_layout(sub, k=0.5, seed=42)
nx.draw_networkx_nodes(sub, pos, node_color=node_colors, node_size=node_sizes, ax=ax)
nx.draw_networkx_edges(sub, pos, edge_color=edge_colors, width=edge_widths, alpha=0.6,
                       arrows=True, arrowsize=8, ax=ax)
# Label only critical-path nodes
cp_labels = {n: str(n)[:18] for n in sub.nodes() if n in cp_set}
nx.draw_networkx_labels(sub, pos, labels=cp_labels, font_size=6, ax=ax)
ax.set_title(f"Build DAG — critical path highlighted ({len(critical_path)} nodes)", fontsize=13)
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ── Bottleneck ranking ───────────────────────────────────────────────
# Rank targets by their impact if their compile time were reduced to zero.
# Impact = compile_time * (1 / (1 + slack))  — high time + low slack = bottleneck.

slack_df["bottleneck_score"] = slack_df["compile_time_s"] / (1.0 + slack_df["slack_s"])
bottleneck_df = slack_df.sort_values("bottleneck_score", ascending=False).head(20)

print("Top-20 bottleneck targets:")
print(bottleneck_df[["target", "compile_time_s", "slack_s", "bottleneck_score"]].to_string(index=False))

# Bar chart
fig, ax = plt.subplots(figsize=(12, 6))
y_pos = range(len(bottleneck_df))
bars = ax.barh(y_pos, bottleneck_df["bottleneck_score"].values, color="tomato", edgecolor="black")
ax.set_yticks(y_pos)
ax.set_yticklabels([str(t)[:30] for t in bottleneck_df["target"].values], fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("Bottleneck score")
ax.set_title("Top-20 build bottlenecks")

# Annotate with compile time
for i, (score, ct) in enumerate(zip(bottleneck_df["bottleneck_score"], bottleneck_df["compile_time_s"])):
    ax.text(score + 0.01 * bottleneck_df["bottleneck_score"].max(), i, f"{ct:.1f}s", va="center", fontsize=7)

plt.tight_layout()
plt.show()

## Codegen on the Critical Path

For each critical-path target, report the fraction of compile time from generated
code and which generators are involved. Also check whether generator execution
steps themselves are on the critical path.

In [ ]:
# ── Codegen on the critical path ─────────────────────────────────────
# For each critical-path target, report what fraction of compile time
# comes from generated code and which generators are involved.

if "is_generated" in file_metrics.columns:
    cp_codegen_records = []
    for target in critical_path:
        target_files = file_metrics[file_metrics["cmake_target"] == target]
        total_ct = target_files["compile_time_ms"].sum() if "compile_time_ms" in target_files.columns else 0
        gen_files = target_files[target_files["is_generated"] == True]
        gen_ct = gen_files["compile_time_ms"].sum() if "compile_time_ms" in gen_files.columns else 0
        generators = sorted(gen_files["generator"].dropna().unique()) if not gen_files.empty else []

        cp_codegen_records.append({
            "target": target,
            "total_compile_ms": total_ct,
            "generated_compile_ms": gen_ct,
            "generated_fraction": gen_ct / total_ct if total_ct > 0 else 0,
            "generators": ", ".join(generators),
        })

    cp_cg_df = pd.DataFrame(cp_codegen_records)
    has_codegen = cp_cg_df[cp_cg_df["generated_fraction"] > 0]

    print(f"Critical path targets with generated code: {len(has_codegen)} / {len(cp_cg_df)}")
    if not has_codegen.empty:
        display(has_codegen.sort_values("generated_fraction", ascending=False))

        fig, ax = plt.subplots(figsize=(12, 5))
        ax.barh(cp_cg_df["target"], cp_cg_df["total_compile_ms"], color="steelblue", label="Hand-written")
        ax.barh(cp_cg_df["target"], cp_cg_df["generated_compile_ms"], color="coral", label="Generated")
        ax.set_xlabel("Compile time (ms)")
        ax.set_title("Critical Path: Generated vs Hand-Written Compile Time")
        ax.invert_yaxis()
        ax.legend()
        plt.tight_layout()
        plt.show()

    # Check if any generator execution steps are on the critical path
    codegen_path = PROJECT_ROOT / "data" / "raw" / "codegen_inventory.csv"
    if codegen_path.exists():
        codegen_inv = pd.read_csv(codegen_path)
        codegen_inv["gen_time_ms"] = pd.to_numeric(codegen_inv["gen_time_ms"], errors="coerce")
        slow_gens = codegen_inv[codegen_inv["gen_time_ms"] > codegen_inv["gen_time_ms"].quantile(0.9)]
        if not slow_gens.empty:
            print("\nSlowest generator invocations (top 10%):")
            display(slow_gens[["generator", "gen_time_ms", "cmake_target", "description"]].sort_values(
                "gen_time_ms", ascending=False).head(10))
else:
    print("is_generated column not available — run codegen inventory first.")